In [1]:
!pip install requests pandas matplotlib textblob


   ---------------------------------------- 0.0/626.3 kB ? eta -:--:--
   --------------------------------------- 626.3/626.3 kB 24.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 19.7 MB/s eta 0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2021.4.4
    Uninstalling regex-2021.4.4:
      Successfully uninstalled regex-2021.4.4
  Attempting uninstall: nltk
    Found existing installation: nltk 3.6.1
    Uninstalling nltk-3.6.1:
      Successfully uninstalled nltk-3.6.1


ERROR: Exception:
Traceback (most recent call last):
  File "d:\anaconda3\lib\site-packages\pip\_internal\cli\base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "d:\anaconda3\lib\site-packages\pip\_internal\cli\base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "d:\anaconda3\lib\site-packages\pip\_internal\cli\req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "d:\anaconda3\lib\site-packages\pip\_internal\commands\install.py", line 484, in run
    installed_versions[distribution.canonical_name] = distribution.version
  File "d:\anaconda3\lib\site-packages\pip\_internal\metadata\pkg_resources.py", line 192, in version
    return parse_version(self._dist.version)
  File "d:\anaconda3\lib\site-packages\pip\_vendor\packaging\version.py", line 56, in parse
    return Version(version)
  File "d:\anaconda3\lib\site-packages\pip\_vendor\packaging\version.py", line 202, in __init__
    raise InvalidVersion(f

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
from datetime import datetime


In [2]:
query = "stranger things"
size = 200

url = "https://api.pushshift.io/reddit/search/comment"

params = {
    "q": query,
    "size": size,
    "fields": "body,author,subreddit,score,created_utc"
}

params


{'q': 'stranger things',
 'size': 200,
 'fields': 'body,author,subreddit,score,created_utc'}

In [3]:
try:
    response = requests.get(url, params=params, timeout=30)
    print("Status code:", response.status_code)
    response.raise_for_status()
    
    json_data = response.json()
    data = json_data.get("data", [])
    
    print("Number of records returned:", len(data))

except Exception as e:
    print("Request failed:", e)
    data = []

Status code: 403
Request failed: 403 Client Error: Forbidden for url: https://api.pushshift.io/reddit/search/comment?q=stranger+things&size=200&fields=body%2Cauthor%2Csubreddit%2Cscore%2Ccreated_utc


In [4]:
df = pd.DataFrame(data)
df.head()

""


In [5]:
if not df.empty:
    df = df[df["body"].notna()]
    df = df[~df["body"].isin(["[deleted]", "[removed]"])]
    df = df.drop_duplicates(subset=["body"]).copy()

    print("Rows after cleaning:", len(df))
else:
    print("DataFrame is empty.")

DataFrame is empty.


In [6]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob

# 1. Query setup
query = "stranger things"
url = "https://api.pushshift.io/reddit/search/comment"

params = {
    "q": query,
    "size": 200,
    "fields": "body,author,subreddit,score,created_utc"
}

# 2. Request data
try:
    response = requests.get(url, params=params, timeout=30)
    print("Status code:", response.status_code)
    response.raise_for_status()
    data = response.json().get("data", [])
    print("Returned rows:", len(data))
except Exception as e:
    print("Request failed:", e)
    data = []

# 3. Convert to DataFrame
df = pd.DataFrame(data)

if not df.empty:
    # 4. Clean data
    df = df[df["body"].notna()]
    df = df[~df["body"].isin(["[deleted]", "[removed]"])]
    df = df.drop_duplicates(subset=["body"]).copy()

    # 5. Time conversion
    df["created_datetime"] = pd.to_datetime(df["created_utc"], unit="s")

    # 6. Sentiment analysis
    def get_sentiment(text):
        polarity = TextBlob(str(text)).sentiment.polarity
        if polarity > 0.1:
            return "Positive"
        elif polarity < -0.1:
            return "Negative"
        else:
            return "Neutral"

    df["sentiment"] = df["body"].apply(get_sentiment)

    # 7. Save CSV
    df.to_csv("stranger_things_comments.csv", index=False, encoding="utf-8-sig")
    print("Saved file: stranger_things_comments.csv")

    # 8. Summary
    print("\nSentiment counts:")
    print(df["sentiment"].value_counts())

    # 9. Plot
    sentiment_counts = df["sentiment"].value_counts()

    plt.figure(figsize=(8, 5))
    plt.bar(sentiment_counts.index, sentiment_counts.values)
    plt.title("Sentiment Distribution of Stranger Things Reddit Comments")
    plt.xlabel("Sentiment")
    plt.ylabel("Number of Comments")
    plt.show()

    # 10. Preview
    display(df.head())

else:
    print("No data returned. Try broadening the query.")

Status code: 403
Request failed: 403 Client Error: Forbidden for url: https://api.pushshift.io/reddit/search/comment?q=stranger+things&size=200&fields=body%2Cauthor%2Csubreddit%2Cscore%2Ccreated_utc
No data returned. Try broadening the query.


In [7]:
import requests
import pandas as pd

# 用PullPush（Pushshift镜像）抓取r/StrangerThings数据
url = "https://api.pullpush.io/reddit/search/submission"
params = {
    "subreddit": "StrangerThings",
    "size": 100,
    "q": "season"
}
response = requests.get(url, params=params)
data = response.json()
df = pd.DataFrame(data['data'])
print(df[['title', 'score', 'num_comments', 'created_utc']].head())

                                               title  score  num_comments  \
0             Who do you think will die in season 5?      1             1   
1  stranger things season 5 if it was based off t...      1             1   
2       My problem with deaths throughout the series      1             1   
3                        Books like stranger things.      1             1   
4  Waiting for the new stranger things season doe...      1             1   

    created_utc  
0  1.746784e+09  
1  1.746748e+09  
2  1.746739e+09  
3  1.746739e+09  
4  1.746737e+09  
